# Step 3: Exploratory Data Analysis (EDA) on IMDb Movie Reviews

**Project:** Aspect-Based Sentiment Analysis on Movies  
**Objective:** Explore the statistical properties, length distributions, class balance, and lexical traits of movie reviews.

---
### EDA Objectives:
1. **Class Distribution Check:** Verify label balance (Positive vs Negative).
2. **Review Length Analysis:** Check word and character counts (critical for transformer sequence length / max_length selection).
3. **Lexical Patterns & Keywords:** Analyze dominant words in positive vs negative reviews.
4. **Missing & Duplicate Detection:** Clean data integrity verification.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure project root in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_preprocessing import load_imdb_raw, get_dataset_summary, get_preprocessed_dataframe, generate_eda_visualizations

print(f"Project Root: {project_root}")

## 1. Load Dataset & Summary Statistics

In [ ]:
dataset = load_imdb_raw()
summary = get_dataset_summary(dataset)
summary_df = pd.DataFrame(summary).T
print("=== COMPREHENSIVE DATASET SUMMARY ===")
display(summary_df)

## 2. Generate and Render EDA Visualizations Suite

In [ ]:
# Generate all 4 high-resolution EDA figures into results/figures/
figures_dir = os.path.join(project_root, "results", "figures")
saved_figures = generate_eda_visualizations(dataset, output_dir=figures_dir)

for name, path in saved_figures.items():
    print(f"\n--- {name.upper().replace('_', ' ')} ---")
    display(Image(filename=path))

## 3. Review Length Quantiles for Transformer Max Length
Transformers such as BERT / DistilBERT have a maximum token length limit (e.g. 512). Understanding the word count percentiles helps determine optimal truncation thresholds.

In [ ]:
train_df = get_preprocessed_dataframe(dataset, split="train")

quantiles = [0.50, 0.75, 0.90, 0.95, 0.99]
word_percentiles = train_df['word_count'].quantile(quantiles)
char_percentiles = train_df['char_length'].quantile(quantiles)

pct_df = pd.DataFrame({
    'Percentile': [f"{int(q*100)}th" for q in quantiles],
    'Word Count': word_percentiles.values.astype(int),
    'Character Length': char_percentiles.values.astype(int)
})

print("=== REVIEW LENGTH PERCENTILES (TRAIN SET) ===")
display(pct_df)

reviews_under_256 = (train_df['word_count'] <= 256).mean() * 100
reviews_under_384 = (train_df['word_count'] <= 384).mean() * 100
print(f"Reviews with <= 256 words: {reviews_under_256:.2f}%")
print(f"Reviews with <= 384 words: {reviews_under_384:.2f}%")